In [ ]:
from pathlib import Path
import os
from os import listdir as ls
import arviz as az
import pandas as pd
import pycountry
import pycountry_convert as pc
from matplotlib_inline.backend_inline import set_matplotlib_formats
import seaborn as sns

from emu_renewal.constants import OUTPUTS_PATH
from emu_renewal.plotting import get_standard_subplot
from emu_renewal.utils import get_countries_by_continent

set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "47213985"
all_countries = ls(job_path)
countries_by_cont = get_countries_by_continent(all_countries)
total_lls = {}
countries = countries_by_cont["AS"][:12]
fig, axes = get_standard_subplot(len(countries) + 1, 4)
flat_axes = axes.ravel()
for c, country in enumerate(countries):
    analyses = {d.name: Path(d.path) for d in os.scandir(job_path / country) if d.is_dir()}
    no_mob_path = analyses["no_mob"]
    idatas = {a: az.from_netcdf(p / "idata_filtered.nc") for a, p in analyses.items()}
    total_lls[country] = pd.DataFrame(columns=analyses)
    ax = flat_axes[c]
    for analysis in analyses:
        idata = idatas[analysis]
        likes = [ll.to_dataframe() for ll in idata.log_likelihood.values()]
        total_lls[country][analysis] = pd.concat(likes, axis=1).sum(axis=1)
    sns.kdeplot(total_lls[country], fill=0.1, ax=ax)
    ax.set_title(country)
fig.tight_layout()